In [1]:
# !pip install librosa soundfile noisereduce hmmlearn praat-parselmouth

In [2]:
import os
import numpy as np
import librosa
import soundfile as sf
import noisereduce as nr
import matplotlib.pyplot as plt

from scipy.signal import butter, filtfilt
from hmmlearn import hmm
from hmmlearn.hmm import GaussianHMM


import re
from tqdm.auto import tqdm
import pickle
import pandas as pd
import ast
from collections import defaultdict

In [3]:
def load_audio(path, sr=16000):
    audio, _ = librosa.load(path, sr=sr, mono=True)
    audio = audio / (np.max(np.abs(audio)) + 1e-8)
    return audio, sr

In [4]:
def bandpass_filter(signal, low=300, high=4500, fs=16000, order=4):
    nyq = fs / 2
    low /= nyq
    high /= nyq
    b, a = butter(order, [low, high], btype="band")
    return filtfilt(b, a, signal)

In [5]:
def reduce_noise_safe(signal, sr):
    noise = signal[:int(0.3 * sr)]
    return nr.reduce_noise(
        y=signal,
        y_noise=noise,
        sr=sr,
        prop_decrease=0.7
    )

In [6]:
def trim_silence(signal):
    trimmed, _ = librosa.effects.trim(
        signal,
        top_db=35,
        frame_length=2048,
        hop_length=512
    )
    return trimmed

In [7]:
def extract_mfcc(signal, sr):
    mfcc = librosa.feature.mfcc(
        y=signal,
        sr=sr,
        n_mfcc=13,
        n_fft=400,
        hop_length=160
    )
    delta = librosa.feature.delta(mfcc)
    delta2 = librosa.feature.delta(mfcc, order=2)
    return np.vstack([mfcc, delta, delta2]).T

In [8]:
def cmvn(features):
    mean = np.mean(features, axis=0)
    std = np.std(features, axis=0) + 1e-8
    return (features - mean) / std

In [9]:
def preprocess_audio(path):
    audio, sr = load_audio(path)
    audio = bandpass_filter(audio)
    audio = reduce_noise_safe(audio, sr)
    audio = trim_silence(audio)
    features = extract_mfcc(audio, sr)
    features = cmvn(features)
    return features

In [10]:
features= preprocess_audio('voice_example/1.mp3')

In [11]:
features.shape

(4698, 39)

In [12]:
def normalize_arabic(text):
    """
    Normalize Arabic text for Qur'an phonemization
    (keep tashkeel, normalize letters)
    """
    text = text.strip()

    # Normalize alef variants
    text = re.sub("[إأآٱ]", "ا", text)

    # Normalize alef maqsoora
    text = re.sub("ى", "ي", text)

    # Normalize taa marbouta
    text = re.sub("ة", "ه", text)

    # Remove tatweel
    text = re.sub("ـ", "", text)

    return text


In [13]:
CONSONANT_MAP = {
    "ب": "b",
    "ت": "t",
    "ث": "θ",
    "ج": "dʒ",
    "ح": "ħ",
    "خ": "x",
    "د": "d",
    "ذ": "ð",
    "ر": "r",
    "ز": "z",
    "س": "s",
    "ش": "ʃ",
    "ص": "sˤ",
    "ض": "dˤ",
    "ط": "tˤ",
    "ظ": "ðˤ",
    "ع": "ʕ",
    "غ": "ɣ",
    "ف": "f",
    "ق": "q",
    "ك": "k",
    "ل": "l",
    "م": "m",
    "ن": "n",
    "ه": "h",
    "و": "w",
    "ي": "j",
    "ء": "ʔ",
}

VOWEL_MAP = {
    "َ": "a",
    "ِ": "i",
    "ُ": "u",
}

SHADDA = "ّ"
SUKUN = "ْ"
DAGGER_ALEF = "ٰ"



In [14]:
def arabic_to_phonemes(text):
    text = normalize_arabic(text)
    phonemes = []

    i = 0

    # Hamzat-wasl for definite article "ال"
    if text.startswith("ال"):
        phonemes.append("a")

    while i < len(text):
        char = text[i]

        # Skip spaces
        if char.isspace():
            i += 1
            continue

        # -----------------
        # Consonants
        # -----------------
        if char in CONSONANT_MAP:
            cons = CONSONANT_MAP[char]

            next_char = text[i+1] if i+1 < len(text) else ""

            # Shadda → double consonant
            if next_char == SHADDA:
                phonemes.append(cons)
                phonemes.append(cons)
                i += 2
                continue

            phonemes.append(cons)
            i += 1
            continue

        # -----------------
        # Short vowels
        # -----------------
        if char in VOWEL_MAP:
            vowel = VOWEL_MAP[char]
            next_char = text[i+1] if i+1 < len(text) else ""

            # Dagger alef (ٰ) → long a
            if next_char == DAGGER_ALEF:
                phonemes.append("aː")
                i += 2
                continue

            # Long vowels
            if vowel == "a" and next_char == "ا":
                phonemes.append("aː")
                i += 2
                continue
            if vowel == "i" and next_char == "ي":
                phonemes.append("iː")
                i += 2
                continue
            if vowel == "u" and next_char == "و":
                phonemes.append("uː")
                i += 2
                continue

            phonemes.append(vowel)
            i += 1
            continue

        # Ignore sukun and unknown symbols
        i += 1

    return phonemes


In [15]:
text = "الرَّحْمَٰنِ"
print(arabic_to_phonemes(text))


['a', 'l', 'r', 'r', 'a', 'ħ', 'm', 'aː', 'n', 'i']


In [16]:
text = "إِنَّا"
print(arabic_to_phonemes(text))


['i', 'n', 'n', 'aː']


In [17]:
print(arabic_to_phonemes("قَالَ"))


['q', 'aː', 'l', 'a']


In [18]:
def phonemes_to_states(phonemes, n_states=3):
    """
    Expand phoneme sequence into HMM state labels
    """
    states = []
    for p in phonemes:
        for i in range(1, n_states + 1):
            states.append(f"{p}_{i}") 
    return states


In [19]:
test_ph = ['a', 'l', 'r']
print(phonemes_to_states(test_ph))


['a_1', 'a_2', 'a_3', 'l_1', 'l_2', 'l_3', 'r_1', 'r_2', 'r_3']


In [20]:
# df = pd.read_csv("datasets/ayat_with_phonemes.csv")

# training_data = []

# for _, row in tqdm(df.iterrows()):

#     # pick one reciter for now (we'll extend later)
#     audio_path = row["Husary"]
#     if ((not isinstance(audio_path, str)) or audio_path =='0.0'):
#         continue

#     phonemes = eval(row["phoneme_sequence"])
#     states = phonemes_to_states(phonemes)
#     mfcc = preprocess_audio(audio_path)

#     training_data.append({
#         "mfcc": mfcc,
#         "states": states
#     })

# with open("training_data.pkl", "wb") as f:
#     pickle.dump(training_data, f)


In [21]:
with open("training_data.pkl", "rb") as f:
    training_data = pickle.load(f)

In [22]:
s=training_data[0]

p = [s.split("_")[0] for s in s["states"]][::3]

print(p)

['b', 'i', 's', 'm', 'i', 'l', 'l', 'l', 'a', 'h', 'i', 'l', 'r', 'r', 'a', 'ħ', 'm', 'aː', 'n', 'i', 'l', 'r', 'r', 'a', 'ħ', 'iː', 'm', 'i', 'q', 'a', 'd', 's', 'a', 'm', 'i', 'ʕ', 'a', 'l', 'l', 'l', 'a', 'h', 'u', 'q', 'a', 'w', 'l', 'a', 'l', 'l', 'a', 't', 'iː', 't', 'u', 'dʒ', 'aː', 'd', 'i', 'l', 'u', 'k', 'a', 'f', 'iː', 'z', 'a', 'w', 'dʒ', 'i', 'h', 'aː', 'w', 'a', 't', 'a', 'ʃ', 't', 'a', 'k', 'iː', 'i', 'l', 'a', 'j', 'l', 'l', 'l', 'a', 'h', 'i', 'w', 'aː', 'l', 'l', 'l', 'a', 'h', 'u', 'j', 'a', 's', 'm', 'a', 'ʕ', 'u', 't', 'a', 'ħ', 'aː', 'w', 'u', 'r', 'a', 'k', 'u', 'm', 'aː', 'i', 'n', 'n', 'a', 'l', 'l', 'l', 'a', 'h', 'a', 's', 'a', 'm', 'iː', 'ʕ', 'b', 'a', 'sˤ', 'iː', 'r']


In [23]:
phoneme_frames = defaultdict(list)

for sample in training_data:
    mfcc = sample["mfcc"]
    phonemes = [s.split("_")[0] for s in sample["states"]][::3]
    # ↑ get phonemes back from states

    T = len(mfcc)
    K = len(phonemes)
    frames_per_ph = T // K

    for i, ph in enumerate(phonemes):
        start = i * frames_per_ph
        end = (i + 1) * frames_per_ph if i < K - 1 else T
        phoneme_frames[ph].append(mfcc[start:end])

In [24]:
# phoneme_models = {}

# for ph, frames_list in tqdm(phoneme_frames.items()):

#     # 🔴 Skip phonemes with no data
#     if len(frames_list) == 0:
#         print(f"⚠️ Skipping phoneme '{ph}' (no samples)")
#         continue

#     X = np.vstack(frames_list)
#     lengths = [len(f) for f in frames_list]

#     # 🔴 Safety check: enough data
#     if X.shape[0] < 10:
#         print(f"⚠️ Skipping phoneme '{ph}' (too little data)")
#         continue

#     model = GaussianHMM(
#         n_components=3,
#         covariance_type="full",
#         n_iter=20,
#         tol=1e-3,
#         init_params="stmc"
#     )

#     model.fit(X, lengths)
#     phoneme_models[ph] = model

# print(f"✅ Trained {len(phoneme_models)} phoneme HMMs")

# with open("phoneme_HMMs.pkl", "wb") as f:
#     pickle.dump(phoneme_models, f)


In [25]:
with open("phoneme_HMMs.pkl", "rb") as f:
    phoneme_models = pickle.load(f)

In [26]:
def build_ayah_states(phonemes, n_states=3):
    """
    Build full HMM state sequence for an ayah
    """
    states = ["sil"]  # start silence

    for p in phonemes:
        for i in range(1, n_states + 1):
            states.append(f"{p}_{i}")

    states.append("sil")  # end silence
    return states


In [27]:
def concatenate_hmms(phoneme_models, phoneme_sequence, n_states=3):
    means = []
    covars = []
    valid_phonemes = []

    for ph in phoneme_sequence:
        if ph not in phoneme_models:
            continue
        means.append(phoneme_models[ph].means_)
        covars.append(phoneme_models[ph].covars_)
        valid_phonemes.append(ph)

    if len(valid_phonemes) == 0:
        raise ValueError("No valid phonemes for this ayah")

    means = np.vstack(means)
    covars = np.vstack(covars)

    total_states = len(valid_phonemes) * n_states
    transmat = np.zeros((total_states, total_states))

    for i in range(total_states):
        if i == total_states - 1:
            # FINAL STATE: must sum to 1
            transmat[i, i] = 1.0
        else:
            transmat[i, i] = 0.6
            transmat[i, i + 1] = 0.4

    hmm = GaussianHMM(
        n_components=total_states,
        covariance_type="full",
        init_params=""
    )

    hmm.startprob_ = np.zeros(total_states)
    hmm.startprob_[0] = 1.0
    hmm.transmat_ = transmat
    hmm.means_ = means
    hmm.covars_ = covars

    return hmm, valid_phonemes


In [28]:
def forced_alignment(hmm, mfcc, phonemes, sr, hop_length=160, n_states=3):
    logprob, states = hmm.decode(mfcc, algorithm="viterbi")

    durations = {}

    for i, ph in enumerate(phonemes):
        start_state = i * n_states
        end_state = start_state + n_states - 1

        frames = np.where(
            (states >= start_state) & (states <= end_state)
        )[0]

        if len(frames) == 0:
            durations[ph] = 0.0
        else:
            durations[ph] = len(frames) * hop_length / sr * 1000

    return durations


In [29]:
def check_madd(durations, min_ms=150):
    errors = []
    for ph, dur in durations.items():
        if ph.endswith("ː") and dur < min_ms:
            errors.append(f"❌ Madd too short on '{ph}' ({dur:.1f} ms)")
    return errors


In [30]:
def check_shadda(phonemes, durations, min_total_ms=80):
    errors = []
    i = 0
    while i < len(phonemes) - 1:
        if phonemes[i] == phonemes[i + 1]:
            total = durations.get(phonemes[i], 0) + durations.get(phonemes[i + 1], 0)
            if total < min_total_ms:
                errors.append(f"❌ Weak or missing shadda on '{phonemes[i]}'")
            i += 2
        else:
            i += 1
    return errors


In [31]:
def check_ghunnah(durations, min_ms=120):
    errors = []
    for ph in ["n", "m"]:
        if ph in durations and durations[ph] < min_ms:
            errors.append(
                f"❌ Ghunnah too short on '{ph}' ({durations[ph]:.1f} ms)"
            )
    return errors


In [32]:
MIN_DURATION = {
    "r": 10,
    "l": 10,
    "n": 20,
    "m": 20
}

def check_missing(durations):
    errors = []
    for ph, dur in durations.items():
        min_dur = MIN_DURATION.get(ph, 20)
        if dur < min_dur:
            errors.append(f"❌ Phoneme '{ph}' likely missing")
    return errors


In [33]:
def evaluate_tajweed(phonemes, durations):
    errors = []
    errors += check_missing(durations)
    errors += check_madd(durations)
    errors += check_ghunnah(durations)
    errors += check_shadda(phonemes, durations)

    if not errors:
        return ["✅ Tajwīd is correct"]
    return errors


In [34]:
df = pd.read_csv("datasets/ayat_with_phonemes.csv")

# Take one ayah (example: first row)
row = df.iloc[215]

print(row['ayah_text'])

# phoneme sequence (stored as string → convert to list)
phonemes = ast.literal_eval(row["phoneme_sequence"])

audio_path = row["Husary"]  # example column
features= preprocess_audio(audio_path)

ayah_hmm, valid_phonemes = concatenate_hmms(
    phoneme_models,
    phonemes,
    n_states=3
)

durations = forced_alignment(
    ayah_hmm,
    features,
    valid_phonemes,
    sr=16000,
    hop_length=160,
    n_states=3
)

evaluate_tajweed(
    phonemes=valid_phonemes,
    durations=durations
)

يَوْمَ يَبْعَثُهُمُ اللَّهُ جَمِيعًا فَيُنَبِّئُهُم بِمَا عَمِلُوا ۚ أَحْصَاهُ اللَّهُ وَنَسُوهُ ۚ وَاللَّهُ عَلَىٰ كُلِّ شَيْءٍ شَهِيدٌ


["❌ Weak or missing shadda on 'b'"]

In [35]:
print("Phoneme durations (ms):")
for ph, d in durations.items():
    print(f"{ph:4s} → {d:.1f} ms")

Phoneme durations (ms):
j    → 110.0 ms
a    → 220.0 ms
w    → 2040.0 ms
m    → 510.0 ms
b    → 30.0 ms
ʕ    → 550.0 ms
θ    → 30.0 ms
u    → 210.0 ms
h    → 30.0 ms
l    → 50.0 ms
dʒ   → 330.0 ms
iː   → 1360.0 ms
f    → 290.0 ms
n    → 140.0 ms
i    → 440.0 ms
aː   → 510.0 ms
uː   → 1510.0 ms
ħ    → 260.0 ms
sˤ   → 520.0 ms
s    → 30.0 ms
k    → 90.0 ms
ʃ    → 50.0 ms
ʔ    → 380.0 ms
d    → 320.0 ms
